In [20]:
import pandas as pd
import numpy as np
import zipfile as zipfile
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#from surprise.model_selection import train_test_split
#import matplotlib.pyplot as plt
#%matplotlib inline
#import sys
#import pickle
#import requests
#from numpy import genfromtxt

## Preprocess

### Movie Data Lens content

In [21]:
# Load movie dataset
mdl_content = pd.read_csv('data_processed/mdl_content.csv.zip',compression='zip', sep=',')

print(mdl_content.info())
mdl_content.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7264 entries, 0 to 7263
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  7264 non-null   int64 
 1   title    7264 non-null   object
 2   genres   7264 non-null   object
 3   year     7264 non-null   int64 
 4   imdbId   7264 non-null   int64 
dtypes: int64(3), object(2)
memory usage: 283.9+ KB
None


,movieId,title,genres,year,imdbId
0,4052,Antitrust,Crime Drama Thriller,2001,218817
1,4053,Double Take,Action Comedy,2001,238948
2,4054,Save the Last Dance,Drama Romance,2001,206275
3,4056,"Pledge, The",Crime Drama Mystery Thriller,2001,237572
4,4068,Sugar & Spice,Comedy,2001,186589


### IMDB content

In [22]:
# Load /  A ajouter : chargement direct dpuis url
imdb_content = pd.read_csv('data_processed/imdb_content.csv.zip',compression='zip', sep=',')

print(imdb_content.info())
imdb_content.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7256 entries, 0 to 7255
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   tconst_short     7256 non-null   int64 
 1   primaryName      7256 non-null   object
 2   primaryName_str  6921 non-null   object
dtypes: int64(1), object(2)
memory usage: 170.2+ KB
None


,tconst_short,primaryName,primaryName_str
0,35423,['James Mangold'],James Mangold
1,96056,"['Menahem Golan', nan]",Menahem Golan nan
2,118141,['Crispin Glover'],Crispin Glover
3,118589,['Vondie Curtis-Hall'],Vondie Curtis-Hall
4,118840,"['Jim Van Bebber', nan, nan]",Jim Van Bebber nan nan


### Merge

In [23]:
df = mdl_content.merge(right=imdb_content, left_on='imdbId', right_on='tconst_short', how='left')

df = df.reset_index().drop(['index'], axis=1)
print(df.info())
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7264 entries, 0 to 7263
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   movieId          7264 non-null   int64  
 1   title            7264 non-null   object 
 2   genres           7264 non-null   object 
 3   year             7264 non-null   int64  
 4   imdbId           7264 non-null   int64  
 5   tconst_short     7256 non-null   float64
 6   primaryName      7256 non-null   object 
 7   primaryName_str  6921 non-null   object 
dtypes: float64(1), int64(3), object(4)
memory usage: 454.1+ KB
None


,movieId,title,genres,year,imdbId,tconst_short,primaryName,primaryName_str
0,4052,Antitrust,Crime Drama Thriller,2001,218817,218817.0,['Peter Howitt'],Peter Howitt
1,4053,Double Take,Action Comedy,2001,238948,238948.0,"['George Gallo', nan, nan]",George Gallo nan nan
2,4054,Save the Last Dance,Drama Romance,2001,206275,206275.0,['Thomas Carter'],Thomas Carter
3,4056,"Pledge, The",Crime Drama Mystery Thriller,2001,237572,237572.0,"['Sean Penn', 'Rajendra Prasad Das', nan]",Sean Penn Rajendra Prasad Das nan
4,4068,Sugar & Spice,Comedy,2001,186589,186589.0,['Francine McDougall'],Francine McDougall


In [24]:
df.duplicated().sum()

0

In [25]:
df.isna().sum()

movieId              0
title                0
genres               0
year                 0
imdbId               0
tconst_short         8
primaryName          8
primaryName_str    343
dtype: int64

### Features build

In [26]:
df['year'] = df['year'].astype('str')
df['primaryName_str'] = df['primaryName_str'].astype('str')

df['combined_features'] = df[['genres', 'year', 'primaryName_str']].apply(lambda x: ' '.join(x), axis=1)

print(df.info())
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7264 entries, 0 to 7263
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movieId            7264 non-null   int64  
 1   title              7264 non-null   object 
 2   genres             7264 non-null   object 
 3   year               7264 non-null   object 
 4   imdbId             7264 non-null   int64  
 5   tconst_short       7256 non-null   float64
 6   primaryName        7256 non-null   object 
 7   primaryName_str    7264 non-null   object 
 8   combined_features  7264 non-null   object 
dtypes: float64(1), int64(2), object(6)
memory usage: 510.9+ KB
None


,movieId,title,genres,year,imdbId,tconst_short,primaryName,primaryName_str,combined_features
0,4052,Antitrust,Crime Drama Thriller,2001,218817,218817.0,['Peter Howitt'],Peter Howitt,Crime Drama Thriller 2001 Peter Howitt
1,4053,Double Take,Action Comedy,2001,238948,238948.0,"['George Gallo', nan, nan]",George Gallo nan nan,Action Comedy 2001 George Gallo nan nan
2,4054,Save the Last Dance,Drama Romance,2001,206275,206275.0,['Thomas Carter'],Thomas Carter,Drama Romance 2001 Thomas Carter
3,4056,"Pledge, The",Crime Drama Mystery Thriller,2001,237572,237572.0,"['Sean Penn', 'Rajendra Prasad Das', nan]",Sean Penn Rajendra Prasad Das nan,Crime Drama Mystery Thriller 2001 Sean Penn Ra...
4,4068,Sugar & Spice,Comedy,2001,186589,186589.0,['Francine McDougall'],Francine McDougall,Comedy 2001 Francine McDougall


### Tokenization / Features extraction

In [27]:
cv = CountVectorizer()
count_matrix = cv.fit_transform(df["combined_features"])
print("Count Matrix:", count_matrix.toarray())

Count Matrix: [[1 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]]


## Model

In [28]:
cosine_sim = cosine_similarity(count_matrix)

In [29]:
print("Min value : ",cosine_sim.min(),"\nMax value : " ,cosine_sim.max(),"\nAverage value : " ,cosine_sim.mean(),"\nMedian value : " ,np.median(cosine_sim))

cosine_sim[:9,:9]

Min value :  0.0 
Max value :  1.0000000000000002 
Average value :  0.13435148694951032 
Median value :  0.14433756729740646


array([[1.        , 0.13608276, 0.36514837, 0.49236596, 0.20412415,
        0.18257419, 0.3086067 , 0.18257419, 0.13608276],
       [0.13608276, 1.        , 0.1490712 , 0.30151134, 0.33333333,
        0.2981424 , 0.12598816, 0.2981424 , 0.11111111],
       [0.36514837, 0.1490712 , 1.        , 0.26967994, 0.2236068 ,
        0.4       , 0.50709255, 0.4       , 0.1490712 ],
       [0.49236596, 0.30151134, 0.26967994, 1.        , 0.15075567,
        0.13483997, 0.22792115, 0.13483997, 0.20100756],
       [0.20412415, 0.33333333, 0.2236068 , 0.15075567, 1.        ,
        0.4472136 , 0.18898224, 0.4472136 , 0.16666667],
       [0.18257419, 0.2981424 , 0.4       , 0.13483997, 0.4472136 ,
        1.        , 0.50709255, 0.6       , 0.1490712 ],
       [0.3086067 , 0.12598816, 0.50709255, 0.22792115, 0.18898224,
        0.50709255, 1.        , 0.3380617 , 0.12598816],
       [0.18257419, 0.2981424 , 0.4       , 0.13483997, 0.4472136 ,
        0.6       , 0.3380617 , 1.        , 0.1490712 ],


### Save of the model

## Test

In [30]:
movie_user_likes = "Antitrust"

def get_index_from_title(title):
    return df[df.title == title]["movieId"].values[0]
    
movie_index = get_index_from_title(movie_user_likes)

movie_index

4052

In [31]:
similar_movies = list(enumerate(cosine_sim[movie_index]))

sorted_similar_movies = sorted(similar_movies, key=lambda x:x[1], reverse=True)[:10]

sorted_similar_movies

[(4052, 1.0),
 (5730, 0.5773502691896258),
 (2163, 0.5),
 (2295, 0.5),
 (2379, 0.5),
 (2387, 0.5),
 (2406, 0.5),
 (2417, 0.5),
 (2427, 0.5),
 (2443, 0.5)]

In [32]:
def get_title_from_index(index):
    return df.iloc[index][1]

print("index 4052 : ",get_title_from_index(4052)) 

index 4052 :  Love Sick (Legaturi bolnavicioase)


In [33]:
for movie in sorted_similar_movies:
    #print(movie[0])
    #index = movie[0]
    #print(index)
    #print(type(index)
    print(get_title_from_index(movie[0]))

Love Sick (Legaturi bolnavicioase)
One to Another (Chacun sa nuit)
Annapolis
Peaceful Warrior
World Trade Center
Pursuit of Happyness, The
Half Nelson
Queen, The
Puffy Chair, The
SherryBaby
